In [1]:
import pandas as pd
import numpy as np
import os

# 1. Membaca contoh isi HI-Small_Patterns.txt untuk memahami tipologi pencucian uang
patterns_path = '../data/raw/HI-Small_Patterns.txt'
with open(patterns_path, 'r') as f:
    lines = [next(f).strip() for _ in range(25)]

print("=== Cuplikan Dokumentasi Pola Pencucian Uang (HI-Small_Patterns.txt) ===")
for line in lines:
    if line:
        print(line)

=== Cuplikan Dokumentasi Pola Pencucian Uang (HI-Small_Patterns.txt) ===
BEGIN LAUNDERING ATTEMPT - FAN-OUT:  Max 16-degree Fan-Out
2022/09/01 00:06,021174,800737690,012,80011F990,2848.96,Euro,2848.96,Euro,ACH,1
2022/09/01 04:33,021174,800737690,020,80020C5B0,8630.40,Euro,8630.40,Euro,ACH,1
2022/09/01 09:14,021174,800737690,020,80006A5E0,35642.49,Yuan,35642.49,Yuan,ACH,1
2022/09/01 09:56,021174,800737690,00220,8007A5B70,5738987.96,US Dollar,5738987.96,US Dollar,ACH,1
2022/09/01 11:28,021174,800737690,001244,80093C0D0,7254.53,US Dollar,7254.53,US Dollar,ACH,1
2022/09/01 13:13,021174,800737690,00513,80078E200,6990.87,US Dollar,6990.87,US Dollar,ACH,1
2022/09/01 14:11,021174,800737690,020,80066B990,12536.92,Euro,12536.92,Euro,ACH,1
2022/09/02 15:40,021174,800737690,00410,8002CC310,3511.82,Euro,3511.82,Euro,ACH,1
2022/09/02 21:23,021174,800737690,01292,8004030A0,16135.09,US Dollar,16135.09,US Dollar,ACH,1
2022/09/02 23:10,021174,800737690,01601,800578800,12183.28,US Dollar,12183.28,US Doll

In [2]:
# Load Master Rekening (Customer Due Diligence / Entity Mapping)
accounts_df = pd.read_csv('../data/raw/HI-Small_accounts.csv')
print(f"Total Master Rekening (CDD): {len(accounts_df):,} akun")
display(accounts_df.head(3))

# Load Log Transaksi dengan tipe data hemat memori
trans_cols = [
    'Timestamp', 'From_Bank', 'From_Account', 'To_Bank', 'To_Account',
    'Amount_Received', 'Receiving_Currency', 'Amount_Paid', 'Payment_Currency',
    'Payment_Format', 'Is_Laundering'
]

trans_df = pd.read_csv(
    '../data/raw/HI-Small_Trans.csv',
    names=trans_cols,
    header=0,
    dtype={
        'From_Bank': 'str',
        'From_Account': 'str',
        'To_Bank': 'str',
        'To_Account': 'str',
        'Amount_Received': 'float32',
        'Receiving_Currency': 'category',
        'Amount_Paid': 'float32',
        'Payment_Currency': 'category',
        'Payment_Format': 'category',
        'Is_Laundering': 'int8'
    }
)

print(f"\nTotal Log Transaksi: {len(trans_df):,} baris")
print(f"Total Kasus Pencucian Uang (Is_Laundering = 1): {trans_df['Is_Laundering'].sum():,} ({trans_df['Is_Laundering'].mean():.4%})")
display(trans_df.head(3))

Total Master Rekening (CDD): 518,581 akun


,Bank Name,Bank ID,Account Number,Entity ID,Entity Name
0,Portugal Bank #4507,331579,80B779D80,80062E240,Sole Proprietorship #50438
1,Canada Bank #27,210,809D86900,800C998A0,Corporation #33520
2,UK Bank #33,21884,80812BE00,800C47F50,Partnership #35397



Total Log Transaksi: 5,078,345 baris
Total Kasus Pencucian Uang (Is_Laundering = 1): 5,177 (0.1019%)


,Timestamp,From_Bank,From_Account,To_Bank,To_Account,Amount_Received,Receiving_Currency,Amount_Paid,Payment_Currency,Payment_Format,Is_Laundering
0,2022/09/01 00:20,010,8000EBD30,010,8000EBD30,3697.340088,US Dollar,3697.340088,US Dollar,Reinvestment,0
1,2022/09/01 00:20,03208,8000F4580,001,8000F5340,0.010000,US Dollar,0.010000,US Dollar,Cheque,0
2,2022/09/01 00:00,03209,8000F4670,03209,8000F4670,14675.570312,US Dollar,14675.570312,US Dollar,Reinvestment,0


In [3]:
# 1. Join Profil Entitas (CDD) untuk Pengirim dan Penerima
acc_lookup = accounts_df[['Account Number', 'Entity ID', 'Entity Name']].drop_duplicates('Account Number')

trans_df = trans_df.merge(
    acc_lookup.rename(columns={'Account Number': 'From_Account', 'Entity ID': 'From_Entity_ID', 'Entity Name': 'From_Entity_Name'}),
    on='From_Account', how='left'
)
trans_df = trans_df.merge(
    acc_lookup.rename(columns={'Account Number': 'To_Account', 'Entity ID': 'To_Entity_ID', 'Entity Name': 'To_Entity_Name'}),
    on='To_Account', how='left'
)

# 2. Parsing Waktu Transaksi
trans_df['Timestamp'] = pd.to_datetime(trans_df['Timestamp'])
trans_df['Hour'] = trans_df['Timestamp'].dt.hour.astype('int8')
trans_df['DayOfWeek'] = trans_df['Timestamp'].dt.dayofweek.astype('int8')
trans_df['Date_Hour'] = trans_df['Timestamp'].dt.floor('h')

# 3. Indikator Kepatuhan & Struktural AML
# a. Apakah transaksi lintas bank? (Inter-bank vs Intra-bank)
trans_df['Is_Cross_Bank'] = (trans_df['From_Bank'] != trans_df['To_Bank']).astype('int8')

# b. Apakah transaksi lintas mata uang? (FX Conversion Risk)
trans_df['Is_Cross_Currency'] = (trans_df['Payment_Currency'].astype(str) != trans_df['Receiving_Currency'].astype(str)).astype('int8')

# c. Apakah di luar jam operasional standar (18:00 - 06:00)?
trans_df['Is_After_Hours'] = ((trans_df['Hour'] >= 18) | (trans_df['Hour'] <= 6)).astype('int8')

# d. Apakah nominal berada di zona Structuring / Smurfing (misal 8.000 - 10.000 USD atau ekuivalen)?
trans_df['Is_Structuring_Zone'] = ((trans_df['Amount_Paid'] >= 8000) & (trans_df['Amount_Paid'] < 10000)).astype('int8')

# e. Apakah transaksi terjadi antar-rekening milik entitas yang sama (Self-Transfer / Layering)?
trans_df['Is_Same_Entity'] = (
    (trans_df['From_Entity_ID'].notna()) & 
    (trans_df['From_Entity_ID'] == trans_df['To_Entity_ID'])
).astype('int8')

# 4. Agregasi Perilaku (Velocity & Deviation) per Rekening Pengirim
# Kecepatan transaksi rekening pada jam yang sama (Velocity)
hourly_vel = trans_df.groupby(['From_Account', 'Date_Hour'])['Amount_Paid'].transform('count')
trans_df['Sender_Hourly_Tx_Count'] = hourly_vel.astype('int16')

# Rasio nominal transaksi saat ini dibanding rata-rata nominal pengirim (Behavioral Spike)
sender_mean_amt = trans_df.groupby('From_Account')['Amount_Paid'].transform('mean')
trans_df['Amount_to_Sender_Avg_Ratio'] = (trans_df['Amount_Paid'] / (sender_mean_amt + 1e-5)).astype('float32')

# Total frekuensi transaksi masuk pada rekening penerima (Fan-In / Smurfing Collector Indicator)
receiver_in_count = trans_df.groupby('To_Account')['Amount_Received'].transform('count')
trans_df['Receiver_Total_In_Tx'] = receiver_in_count.astype('int32')

print("Feature Engineering AML & CDD selesai!")
display(trans_df[['Timestamp', 'From_Bank', 'To_Bank', 'Amount_Paid', 'Payment_Format', 'Is_Cross_Bank', 'Is_Cross_Currency', 'Sender_Hourly_Tx_Count', 'Amount_to_Sender_Avg_Ratio', 'Is_Laundering']].head(5))

Feature Engineering AML & CDD selesai!


,Timestamp,From_Bank,To_Bank,Amount_Paid,Payment_Format,Is_Cross_Bank,Is_Cross_Currency,Sender_Hourly_Tx_Count,Amount_to_Sender_Avg_Ratio,Is_Laundering
0,2022-09-01 00:20:00,010,010,3697.340088,Reinvestment,0,0,1,1.923694,0
1,2022-09-01 00:20:00,03208,001,0.010000,Cheque,1,0,1,0.000021,0
2,2022-09-01 00:00:00,03209,03209,14675.570312,Reinvestment,0,0,1,1.000000,0
3,2022-09-01 00:02:00,012,012,2806.969971,Reinvestment,0,0,3,0.015411,0
4,2022-09-01 00:06:00,010,010,36682.968750,Reinvestment,0,0,2,0.738840,0


In [4]:
os.makedirs('../data/processed', exist_ok=True)

# Hapus kolom sementara Date_Hour sebelum disimpan
trans_df.drop(columns=['Date_Hour'], inplace=True)

output_file = '../data/processed/aml_transactions_enriched.parquet'
trans_df.to_parquet(output_file, index=False, compression='snappy')

size_mb = os.path.getsize(output_file) / (1024 * 1024)
print(f"Berhasil disimpan ke {output_file} | Ukuran file: {size_mb:.2f} MB")

Berhasil disimpan ke ../data/processed/aml_transactions_enriched.parquet | Ukuran file: 223.72 MB


In [1]:
import pandas as pd
import numpy as np
import os

df = pd.read_parquet('../data/processed/aml_transactions_enriched.parquet')
df = df.sort_values('Timestamp').reset_index(drop=True)

print("Membangun Fitur Graf Jaringan (Fan-In, Fan-Out, Pass-Through, & Reciprocity)...")

# 1. Fan-Out Degree: Jumlah rekening penerima UNIK yang dikirimi dana oleh From_Account
df['Sender_Unique_Receivers'] = df.groupby('From_Account')['To_Account'].transform('nunique').astype('int32')

# 2. Fan-In Degree: Jumlah rekening pengirim UNIK yang mengirim dana ke To_Account
df['Receiver_Unique_Senders'] = df.groupby('To_Account')['From_Account'].transform('nunique').astype('int32')

# 3. Sender Total Outgoing Transactions & Total Volume
df['Sender_Total_Out_Tx'] = df.groupby('From_Account')['Amount_Paid'].transform('count').astype('int32')
df['Sender_Total_Out_Vol'] = df.groupby('From_Account')['Amount_Paid'].transform('sum').astype('float32')

# 4. Receiver Total Incoming Volume & Avg Incoming Amount
df['Receiver_Total_In_Vol'] = df.groupby('To_Account')['Amount_Received'].transform('sum').astype('float32')
receiver_mean_amt = df.groupby('To_Account')['Amount_Received'].transform('mean')
df['Amount_to_Receiver_Avg_Ratio'] = (df['Amount_Received'] / (receiver_mean_amt + 1e-5)).astype('float32')

# 5. Pairwise Frequency (Seberapa sering pasangan Pengirim -> Penerima yang sama bertransaksi)
df['Pair_Tx_Count'] = df.groupby(['From_Account', 'To_Account'])['Amount_Paid'].transform('count').astype('int32')

# 6. Pass-Through Time Delta (Selisih detik sejak transaksi terakhir pada rekening pengirim)
df['Time_Since_Last_Sender_Tx'] = df.groupby('From_Account')['Timestamp'].diff().dt.total_seconds().fillna(999999).astype('float32')

# 7. Pass-Through Account Indicator (Apakah rekening pengirim juga aktif menerima dana dari pihak lain?)
in_vol_map = df.groupby('To_Account')['Amount_Received'].sum()
df['Sender_Prior_In_Vol'] = df['From_Account'].map(in_vol_map).fillna(0).astype('float32')
df['Pass_Through_Ratio'] = (df['Sender_Total_Out_Vol'] / (df['Sender_Prior_In_Vol'] + 1.0)).astype('float32')

# 8. Rasio Fan-In vs Fan-Out (Mendeteksi akun pengumpul / collector vs akun penyebar / distributor)
df['Fan_In_Out_Ratio'] = ((df['Receiver_Unique_Senders'] + 1) / (df['Sender_Unique_Receivers'] + 1)).astype('float32')

output_file = '../data/processed/aml_transactions_enriched.parquet'
df.to_parquet(output_file, index=False, compression='snappy')
print(f"Selesai! Total kolom sekarang: {df.shape[1]} | Ukuran file: {os.path.getsize(output_file)/(1024*1024):.2f} MB")

Membangun Fitur Graf Jaringan (Fan-In, Fan-Out, Pass-Through, & Reciprocity)...
Selesai! Total kolom sekarang: 36 | Ukuran file: 377.99 MB
